# Investigating the Met's Collection
### What Gets Displayed, What Gets Collected, and What Art Reveals About Its Era

DS3000 Final Project

Group: Anna Bennett, Zara Ceraj

**1. (20 pts) Expresses the central motivation of the project and explains the (at least) two key questions to be explored. Gives a summary of the data processing pipeline so a technical expert can easily follow along.**

### Central Motivation and Key Questions
When you walk into a large art museum, it feels like you could spend a week walking through carefully looking at every piece. However, most museums only show a fraction of their collections. For instance, "the Tate shows about 20% of its permanent collection. The Louvre shows 8%, the Guggenheim a lowly 3% and the Berlinische Galerie – a Berlin museum whose mandate is to show, preserve and collect art made in the city – 2% of its holdings" (Bradley, 2015). Physical space is part of the reason, but what ends up on the walls also reflects each institution's curatorial priorities. My questions arise out of these inaccessible collections that many museums have, focusing more specifically on the Metropolitan Museum of Art.

Question 1: What predicts whether a work is on display?

Can features like an artwork's age, size, department, culture of origin, or the artist's gender predict whether it hangs in a gallery or sits in storage?

What gets displayed depends on what was collected in the first place, and collecting has historically been uneven. In 1989, the Guerrilla Girls, an anonymous collective of feminist artists, visited the Met and compared the number of women artists in its modern art galleries with the number of female nudes on display. They found that fewer than 5% of the artists in those galleries were women, while 85% of the nudes were female (Guerrilla Girls, 1989). Decades later, the gap remains: between 2008 and 2018, "works by women constituted just 11 percent of acquisitions and 14 percent of exhibitions at 26 major American museums" (Solly, 2019).

Artistic movements are often defined by new materials, techniques, or philosophies, but there is overlap across very different eras. Oil paint, for example, was used centuries apart by both Renaissance and Impressionist painters.

Question 2: How accurately can an artwork's time period be predicted from its traits, such as its medium, size, and culture of origin?

### Data Processing Pipeline
-  **Get all object IDs:** A GET request to the Met Collection API's /objects endpoint returns the IDs of every object in the collection (~502,000).
- **Sample:** Because the API returns only one object per request, we draw a random sample of 500 IDs (random.seed(3000) for reproducibility).
- **Fetch each object:** For each sampled ID, we request /objects/{id} with a browser User-Agent header. --> A failed request (e.g., a 403 from rate limiting) is retried up to 3 times with increasing waits (5, 10, 15 s), and a 1 s pause follows each successful request. Objects that still fail are skipped and counted.
- **Extract features:** From each JSON record we keep 13 fields: objectID, title, department, classification, culture, medium, artistDisplayName, artistGender, objectBeginDate, objectEndDate, accessionYear, and GalleryNumber. Height and width (cm) are arsed from the nested measurements list, using the element named `"Overall"; they are left null if missing.
- **Build the DataFrame and clean:** The rows are combined into a pandas DataFrame (500 rows × 15 columns).
- **Save:** The result is written to met_sample.csv.

**2. (25 pts) Obtains, cleans, and merges all data sources involved in the project.**

## Hi, note: the data hasn't fully been cleaned yet, theres one date still in the future and some empty data points that idk how we want to handle

### Data Source
The Metropolitan Museum of Art (the Met) in New York City is one of the largest art museums in the world, with a collection spanning thousands of years. The museum publishes data on over 502,000 artworks through a free public API that requires no registration or key.

Each artwork's record includes dozens of fields, such as when the work was made, when the museum acquired it, its dimensions, its department and culture of origin, and whether it is currently on view.

[Data Source: The Metropolitan Museum of Art Collection API](https://metmuseum.github.io/)

In [4]:
import random
import time

import pandas as pd
import requests

In [5]:
url = "https://collectionapi.metmuseum.org/public/collection/v1/objects"

response = requests.get(url)

all_ids = response.json()["objectIDs"]
print("Total objects in the collection:", len(all_ids))

random.seed(3000)
sample_ids = random.sample(all_ids, 500)

Total objects in the collection: 503152


Note: Due to the size of the collection and since the API only returns one object per request, I took a random sample of 500 IDs. This sample might not accurately reflect the full collection, but the API's rate limits made larger-scale collection impractical.

In [6]:
rows = []
failed = 0

for object_id in sample_ids:
    # retry up to 3 times, waiting longer each time, since the API blocks fast repeated requests
    # Per syllabus: I used AI to help me write this chunk as I kept getting 403 errors when I tried to run the code
    # without a user agent and without a retry mechanism.
    response = None
    for attempt in range(3):
        response = requests.get(url + "/" + str(object_id), headers={"User-Agent": "Mozilla/5.0"})
        if response.status_code == 200:
            break
        time.sleep(5 * (attempt + 1))
    if response.status_code != 200:
        failed += 1
        continue
    obj = response.json()

    # find the overall height and width but some have missing data
    height = None
    width = None
    if obj["measurements"] is not None:
        for m in obj["measurements"]:
            if m["elementName"] == "Overall":
                sizes = m["elementMeasurements"]
                if "Height" in sizes:
                    height = sizes["Height"]
                if "Width" in sizes:
                    width = sizes["Width"]

    row = {
        "objectID": obj["objectID"],
        "title": obj["title"],
        "department": obj["department"],
        "classification": obj["classification"],
        "culture": obj["culture"],
        "medium": obj["medium"],
        "artistDisplayName": obj["artistDisplayName"],
        "artistGender": obj["artistGender"],
        "objectBeginDate": obj["objectBeginDate"],
        "objectEndDate": obj["objectEndDate"],
        "accessionYear": obj["accessionYear"],
        "height_cm": height,
        "width_cm": width,
        "GalleryNumber": obj["GalleryNumber"],
    }
    rows.append(row)

    time.sleep(1) # had to keep extending this...

print("Failed requests:", failed)

met_df = pd.DataFrame(rows)
met_df.head()


Failed requests: 0


,objectID,title,department,classification,culture,medium,artistDisplayName,artistGender,objectBeginDate,objectEndDate,accessionYear,height_cm,width_cm,GalleryNumber
0,502517,Bell,Musical Instruments,Idiophone-Struck-bell-clapper,Italian (Ancient Roman),bronze,,,1,1,1889,NaN,NaN,
1,79430,Suit,Costume Institute,,British,synthetic fiber,Biba,,1967,1977,1997,NaN,NaN,
2,268787,[Old Man with Hat],Photographs,Photographs,,Carbon print,Pirie MacDonald,,1900,1919,1943,NaN,NaN,
3,51986,Tile with characters,Asian Art,Ceramics,China,Earthenware,,,-221,220,1910,15.557531,5.715012,
4,249668,Glass pendant shaped like a jar,Greek and Roman Art,Glass,Roman,Glass,,,200,399,1917,NaN,NaN,171


In [7]:
# cleaning and simplifying the data
met_df["onView"] = met_df["GalleryNumber"] != ""
met_df["accessionYear"] = pd.to_numeric(met_df["accessionYear"], errors="coerce")

met_df.to_csv("met_sample.csv", index=False)

print("DataFrame shape:", met_df.shape)

print("\nDescription:")
met_df.describe()
met_df.sample(10)

DataFrame shape: (500, 15)

Description:


,objectID,title,department,classification,culture,medium,artistDisplayName,artistGender,objectBeginDate,objectEndDate,accessionYear,height_cm,width_cm,GalleryNumber,onView
114,464332,Reliquary Arm of St. Valentine,Medieval Art,Metalwork-Silver,Swiss,"Silver, gilded silver, and a sapphire",,,1300,1399,1917.0,55.200000,19.400000,306,True
129,666311,The Virgin seated with the Christ Child on her...,Drawings and Prints,Prints,,Etching,Ventura Salimbeni,,1595,1618,1951.0,NaN,NaN,,False
75,503218,Suldae,Musical Instruments,Chordophone-Accessories-plectrum,Korean,Bamboo,,,1901,2000,1982.0,NaN,NaN,,False
198,452552,Sprinkler Bottle with Fesse Emblem,Islamic Art,Glass,,"Glass, colorless; blown, applied handles, enam...",,,1275,1325,1974.0,8.731268,6.508763,,False
233,61187,Belt Hook,Asian Art,Metalwork,China,Silver,,,960,1279,1926.0,4.800000,2.500000,,False
211,650520,"Plate 7, from ""Recueil de Vases analogues aux ...",Drawings and Prints,Prints,,Etching,Guillaume Martin Couture,,1745,1799,1969.0,NaN,NaN,,False
377,121326,Dress,Costume Institute,,British,"silk, cotton",,,1805,1810,1987.0,NaN,NaN,,False
481,5942,Plate,The American Wing,,,Wood; Maple,,,1700,1900,1910.0,NaN,NaN,,False
283,370577,The House That Glue Built,Drawings and Prints,Books,,Illustrations: lithographs,Clara Andrews Wiliams,,1905,1905,1993.0,26.000000,36.000000,,False
369,22725,Scroll,Arms and Armor,Miscellaneous,Japanese,Ink and pigments on paper,,,1590,1900,1917.0,NaN,NaN,,False


Note: the dates make sense because they used negative dates for BC and positive for AD.

However, one accession year is listed as 2056, which is likely a data entry error I will remove during further cleaning.

**3. (25 pts) Builds at least two visualizations (graphs/plots) from the data which help to understand or answer the questions of interest. These visualizations will be graded based on how much information they can effectively communicate to readers. Please make sure your visualization are sufficiently distinct from each other.**

In [10]:
# calculate display percentage by department and classification
import plotly.express as px

# find the 15 departments with the most artworks
top_departments = met_df["department"].value_counts().head(15).index
plot_data = met_df[met_df["department"].isin(top_departments)].copy()

# count displayed and undisplayed artworks
plot_data["Display Status"] = plot_data["onView"].map({
    True: "On Display",
    False: "Not on Display"})

fig = px.histogram(plot_data,
    x="department",
    color="Display Status",
    barmode="group",
    title="Displayed vs. Undisplayed Artworks by Department",
    labels={"department": "Museum Department",
        "count": "Number of Artworks"})

fig.update_layout(xaxis_tickangle=-45)
fig.show()

**4. (20 pts) Develop at least two functions that will verify fit or other attributes of analysis from a ML model. (You don't need to identify a specific model yet).  Include markup sections which explain why these functions will be useful in analyzing a ML model.**

In [4]:
# Function 1 evaluates the model accuracy
# MAE shows mean distance from each predicted value to the actual
# RMSE highlights outliers
# Rsquared analyzes fit

def evaluateModel(model, X_test, y_test):
  """ Given a model and the test data, evaluate the model's accuracy given 3 metrics. """
  y_pred = model.predict(X_test)

  print("MAE:", mean_absolute_error(y_test, y_pred))
  print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
  print("R²:", r2_score(y_test, y_pred))


Function 1 will evaluate the fit of the model. Using the Mean absolute error, the average distance between the actual and predicted values. It doesn't consider whether some or all values are higher or lower, only their average distance from the mean. The RMSE compares the model to actual data as well, but puts it in the same units to make the comparison more comprehensible and investigate any outliers. R-squared shows goodness of fit and measures the proportion of variance in the dependent variable. All of these features give a more well-rounded picture of the fit of the model.

**5. (10 pts) Discuss potential ethical implications of the data and using ML to analyze it.**

Some ethical implications include sourcing issues where many of the artists are not alive to consent to data about their artwork being analyzed a a machine learning model, or even if they are alright with their data being public at all. The actual use vs the understood use by either the museum or the artists of their art and/or data attirubutes may not align. Additionally, art contains many important contexts and motifs that a machine learning model may not be able to capture which could lead to misrepresentation or misinterpretation of art. Missing information especially from older art could lead to some bias in how the model is able to represent the sample, and historically underrepresented artists may continue to be underrepresented in the model's analysis.

### References

Bradley, K. (2015, January 23). Why museums hide masterpieces away. BBC Culture. [https://www.bbc.com/culture/article/20150123-7-masterpieces-you-cant-see](https://www.bbc.com/culture/article/20150123-7-masterpieces-you-cant-see)

Guerrilla Girls. (1989). Do women have to be naked to get into the Met. Museum? The Metropolitan Museum of Art, New York, NY. [https://www.metmuseum.org/art/collection/search/849438](https://www.metmuseum.org/art/collection/search/849438)

Solly, M. (2019, September 20). Study shows U.S. museums still lag when it comes to acquiring works by women artists. Smithsonian Magazine. [https://www.smithsonianmag.com/smart-news/survey-us-museums-reveals-few-gains-female-artists-past-decade-180973190/](https://www.smithsonianmag.com/smart-news/survey-us-museums-reveals-few-gains-female-artists-past-decade-180973190/)